In [1]:
!pip install -q torch transformers accelerate sentencepiece xgboost

In [1]:
import os, re, gc, json, string, warnings, logging, argparse
from pathlib import Path
from datetime import datetime
from collections import Counter
 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib


from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.semi_supervised import LabelPropagation
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, silhouette_score
)
 
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    warnings.warn("XGBoost not found. Install: !pip install xgboost -q")
 
try:
    import torch
    print("✅ torch imported")

    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from torch.cuda.amp import autocast, GradScaler
    print("✅ torch modules imported")

    import transformers
    print("✅ transformers imported")

    from transformers import (
        BertTokenizerFast,
        BertForSequenceClassification,
        get_linear_schedule_with_warmup,
    )
    print("✅ transformer classes imported")

    from torch.optim import AdamW
    print("✅ AdamW imported")

    TORCH_AVAILABLE = True
    print("🎉 Everything working!")

except Exception as e:
    TORCH_AVAILABLE = False
    print("❌ REAL ERROR:")
    print(type(e).__name__)
    print(e)
 
warnings.filterwarnings("ignore")

✅ torch imported
✅ torch modules imported
✅ transformers imported
✅ transformer classes imported
✅ AdamW imported
🎉 Everything working!


In [ ]:
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from torch.optim import AdamW

In [ ]:
"""
model_a_kaggle.py
=================
Model A — Complete Pipeline for Kaggle
Project: Intelligent Reading Comprehension & Quiz Generation System

Single file combining:
  PHASE 1 — Preprocessing  (preprocessing.py v2)
  PHASE 2 — Classical ML   (model_a_train.py)
  PHASE 3 — BERT           (model_a_bert.py)

Kaggle paths
------------
  Input  : /kaggle/input/datasets/ankitdhiman7/race-dataset/
  Working: /kaggle/working/   (all outputs written here)

Directory layout written to /kaggle/working/
---------------------------------------------
  train_clean.csv / dev_clean.csv / test_clean.csv
  train_ohe.npz   / dev_ohe.npz   / test_ohe.npz
  train_tfidf.npz / dev_tfidf.npz / test_tfidf.npz
  train_cosine_ohe.csv  / train_cosine_tfidf.csv / train_jaccard.csv  (+ dev/test)
  preprocessing_config.json

  models/model_a/traditional/
      lr_tfidf.joblib
      svm_calibrated.joblib
      naive_bayes.joblib
      random_forest.joblib
      xgboost.joblib
      kmeans.joblib  /  kmeans_svd.joblib
      gmm.joblib
      label_propagation.joblib
      stacking_meta_lr.joblib
      ohe_vectorizer.joblib
      tfidf_vectorizer.joblib
      label_encoder.joblib

  models/model_a/bert/
      best_model/         (HuggingFace model dir)
      tokenizer/
      training_log.csv
      bert_config.json

  results/
      model_a_supervised_results.csv
      model_a_unsupervised_results.csv
      model_a_bert_test_result.csv
      model_a_results_<timestamp>.csv
      model_a_test_result.csv

How to run on Kaggle
--------------------
Option A — Run all three phases sequentially (recommended, Internet ON for BERT):
    exec(open("model_a_kaggle.py").read())
    # or in a notebook cell:
    # %run model_a_kaggle.py

Option B — Run phases individually:
    run_preprocessing()
    run_classical_models()
    run_bert(epochs=3)

Option C — Phases via CLI flags when running as script:
    !python model_a_kaggle.py --skip_bert          # preprocessing + classical only
    !python model_a_kaggle.py --skip_classical     # preprocessing + bert only
    !python model_a_kaggle.py --bert_epochs 5
    !python model_a_kaggle.py --quick              # fast debug (20k rows, 1 BERT epoch)
"""

# =============================================================================
# IMPORTS
# =============================================================================
import os, re, gc, json, string, warnings, logging, argparse
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib

from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.semi_supervised import LabelPropagation
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, silhouette_score
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    warnings.warn("XGBoost not found. Install: !pip install xgboost -q")

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from torch.cuda.amp import autocast, GradScaler
    from transformers import (
        BertTokenizerFast, BertForSequenceClassification,
        AdamW, get_linear_schedule_with_warmup,
    )
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    warnings.warn("PyTorch / transformers not found. BERT phase will be skipped.")

warnings.filterwarnings("ignore")

# =============================================================================
# ── KAGGLE PATH CONFIGURATION ─────────────────────────────────────────────────
# =============================================================================
BASE_INPUT  = Path("/kaggle/input")
BASE_WORK   = Path("/kaggle/working")

RAW_DATA_PATH   = BASE_INPUT / "datasets/ankitdhiman7/race-dataset"
PROCESSED_PATH  = BASE_WORK                                  # CSVs + .npz written here
MODELS_TRAD     = BASE_WORK / "models/model_a/traditional"
MODELS_BERT     = BASE_WORK / "models/model_a/bert"
RESULTS_PATH    = BASE_WORK / "results"

for _p in [MODELS_TRAD, MODELS_BERT, RESULTS_PATH]:
    _p.mkdir(parents=True, exist_ok=True)

# =============================================================================
# SHARED CONSTANTS
# =============================================================================
RANDOM_SEED       = 42
N_CLASSES         = 4
CV_FOLDS          = 3
KMEANS_K          = 4
GMM_K             = 4
LABEL_PROP_FRAC   = 0.05
SVD_COMPONENTS    = 200
DEFAULT_MAX_VOCAB = 10_000
JACCARD_TOPK      = 20
BERT_MODEL_NAME   = "bert-base-uncased"
BERT_MAX_LENGTH   = 512
WARMUP_RATIO      = 0.10

STOP_WORDS = {
    "a","an","the","and","or","but","in","on","at","to","for","of","with",
    "is","was","are","were","be","been","being","have","has","had","do",
    "does","did","will","would","could","should","may","might","shall",
    "that","this","these","those","it","its","by","from","as","not","no",
    "so","if","then","than","more","also","very","just","about","i","we",
    "you","he","she","they","them","their","our","my","your","his","her",
    "which","who","whom","what","when","where","why","how","all","each",
    "every","both","few","some","such","into","through","during","before",
    "after","above","below","between","out","off","over","under","again",
    "further","once","here","there","can","up","down","am",
}

DENSE_COLS = [
    "article_word_count","article_char_count",
    "question_word_count","question_char_count",
    "q_article_token_overlap",
    "A_article_overlap","B_article_overlap","C_article_overlap","D_article_overlap",
    "A_question_overlap","B_question_overlap","C_question_overlap","D_question_overlap",
    "A_word_count","B_word_count","C_word_count","D_word_count",
    "correct_option_art_overlap","correct_option_q_overlap",
    "ohe_cos_art_q","ohe_cos_art_A","ohe_cos_art_B","ohe_cos_art_C","ohe_cos_art_D",
    "tfidf_cos_art_q","tfidf_cos_art_A","tfidf_cos_art_B","tfidf_cos_art_C","tfidf_cos_art_D",
    "jaccard_q_A","jaccard_q_B","jaccard_q_C","jaccard_q_D",
    "max_sentence_cos",
]

# =============================================================================
# LOGGING
# =============================================================================
logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S", level=logging.INFO,
)
log = logging.getLogger(__name__)


# #############################################################################
#
#  PHASE 1 — PREPROCESSING
#
# #############################################################################

# ── Text cleaning ─────────────────────────────────────────────────────────────
_PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.encode("ascii", errors="ignore").decode("ascii")
    text = text.translate(_PUNCT_TABLE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    log.info("  Cleaning text columns ...")
    df = df.copy()
    for col in ["article","question","A","B","C","D"]:
        df[col] = df[col].fillna("").astype(str)
    bad = ~df["answer"].isin({"A","B","C","D"})
    if bad.sum():
        log.warning(f"  Dropping {bad.sum()} rows with invalid answer labels.")
        df = df[~bad].reset_index(drop=True)
    for col in ["article","question","A","B","C","D"]:
        df[f"{col}_clean"] = df[col].apply(clean_text)
    empty = (df["article_clean"].str.len()==0)|(df["question_clean"].str.len()==0)
    if empty.sum():
        df = df[~empty].reset_index(drop=True)
    before = len(df)
    df.drop_duplicates(subset=["article","question","answer"], keep="first", inplace=True)
    df.reset_index(drop=True, inplace=True)
    log.info(f"  Rows after cleaning: {len(df):,}  (removed {before-len(df):,} dupes)")
    return df

def build_combined_text(df: pd.DataFrame) -> pd.DataFrame:
    """Article repeated twice for weight boost (Manual §6.2)."""
    df = df.copy()
    df["combined_text"] = (
        df["article_clean"]+" "+df["article_clean"]+" "+
        df["question_clean"]+" "+
        df["A_clean"]+" "+df["B_clean"]+" "+df["C_clean"]+" "+df["D_clean"]
    )
    for opt in ["A","B","C","D"]:
        df[f"combined_{opt}"] = (
            df["article_clean"]+" "+df["article_clean"]+" "+
            df["question_clean"]+" "+df[f"{opt}_clean"]
        )
    return df

# ── Lexical features ──────────────────────────────────────────────────────────
def _token_overlap(a: str, b: str) -> float:
    ta = set(a.split()) - STOP_WORDS
    tb = set(b.split()) - STOP_WORDS
    return len(ta & tb) / len(tb) if tb else 0.0

def build_lexical_features(df: pd.DataFrame) -> pd.DataFrame:
    log.info("  Computing 19 lexical features ...")
    df = df.copy()
    df["article_word_count"]      = df["article_clean"].apply(lambda t: len(t.split()))
    df["article_char_count"]      = df["article_clean"].apply(len)
    df["question_word_count"]     = df["question_clean"].apply(lambda t: len(t.split()))
    df["question_char_count"]     = df["question_clean"].apply(len)
    df["q_article_token_overlap"] = df.apply(
        lambda r: _token_overlap(r["article_clean"], r["question_clean"]), axis=1)
    for opt in ["A","B","C","D"]:
        df[f"{opt}_article_overlap"]  = df.apply(
            lambda r,o=opt: _token_overlap(r["article_clean"],  r[f"{o}_clean"]), axis=1)
        df[f"{opt}_question_overlap"] = df.apply(
            lambda r,o=opt: _token_overlap(r["question_clean"], r[f"{o}_clean"]), axis=1)
        df[f"{opt}_word_count"]       = df[f"{opt}_clean"].apply(lambda t: len(t.split()))
    df["correct_option_art_overlap"] = df.apply(
        lambda r: r[f"{r['answer']}_article_overlap"],  axis=1)
    df["correct_option_q_overlap"]   = df.apply(
        lambda r: r[f"{r['answer']}_question_overlap"], axis=1)
    return df

# ── Vectorizers ───────────────────────────────────────────────────────────────
def fit_ohe_vectorizer(texts: pd.Series, max_vocab: int) -> CountVectorizer:
    log.info(f"  Fitting OHE vectorizer (max_vocab={max_vocab:,}) ...")
    vec = CountVectorizer(
        binary=True, max_features=max_vocab, ngram_range=(1,2),
        min_df=3, max_df=0.95, stop_words=list(STOP_WORDS),
        token_pattern=r"(?u)\b[a-z]{2,}\b",
    )
    vec.fit(texts)
    log.info(f"  OHE vocab: {len(vec.vocabulary_):,}")
    return vec

def fit_tfidf_vectorizer(texts: pd.Series, max_vocab: int) -> TfidfVectorizer:
    log.info(f"  Fitting TF-IDF vectorizer (max_vocab={max_vocab:,}) ...")
    vec = TfidfVectorizer(
        max_features=max_vocab, ngram_range=(1,2), min_df=3, max_df=0.95,
        stop_words=list(STOP_WORDS), sublinear_tf=True, norm="l2",
        token_pattern=r"(?u)\b[a-z]{2,}\b",
    )
    vec.fit(texts)
    log.info(f"  TF-IDF vocab: {len(vec.vocabulary_):,}")
    return vec

def inspect_vocab(vec: TfidfVectorizer, mat: sp.csr_matrix, n: int = 10):
    names  = vec.get_feature_names_out()
    means  = np.asarray(mat.mean(axis=0)).ravel()
    top    = means.argsort()[::-1][:n]
    log.info(f"  Top-{n} TF-IDF terms:")
    for i, idx in enumerate(top, 1):
        log.info(f"    {i:2d}. {names[idx]:35s} {means[idx]:.5f}")

# ── Cosine features ───────────────────────────────────────────────────────────
def _row_cos(mx: sp.csr_matrix, my: sp.csr_matrix) -> np.ndarray:
    nx = np.array(mx.power(2).sum(axis=1)).ravel() ** 0.5
    ny = np.array(my.power(2).sum(axis=1)).ravel() ** 0.5
    dt = np.array(mx.multiply(my).sum(axis=1)).ravel()
    dn = nx * ny
    dn[dn == 0] = 1e-9
    return (dt / dn).astype(np.float32)

def _cosine_block(df, vec, prefix, BATCH=500):
    cols = [f"{prefix}_cos_art_q",f"{prefix}_cos_art_A",
            f"{prefix}_cos_art_B",f"{prefix}_cos_art_C",f"{prefix}_cos_art_D"]
    res  = {c: np.zeros(len(df), dtype=np.float32) for c in cols}
    is_tfidf = isinstance(vec, TfidfVectorizer)
    for s in range(0, len(df), BATCH):
        e = min(s+BATCH, len(df)); b = df.iloc[s:e]
        art = vec.transform(b["article_clean"])
        q   = vec.transform(b["question_clean"])
        if is_tfidf:   # dot == cosine because L2-normalised
            res[f"{prefix}_cos_art_q"][s:e] = np.array(
                art.multiply(q).sum(axis=1)).ravel().astype(np.float32)
        else:
            res[f"{prefix}_cos_art_q"][s:e] = _row_cos(art, q)
        for opt in ["A","B","C","D"]:
            om = vec.transform(b[f"{opt}_clean"])
            if is_tfidf:
                res[f"{prefix}_cos_art_{opt}"][s:e] = np.array(
                    art.multiply(om).sum(axis=1)).ravel().astype(np.float32)
            else:
                res[f"{prefix}_cos_art_{opt}"][s:e] = _row_cos(art, om)
    return pd.DataFrame(res, index=df.index)

# ── Jaccard features ──────────────────────────────────────────────────────────
def _jaccard_topk(vq, vo, k=JACCARD_TOPK):
    tq  = set(np.argpartition(vq, -k)[-k:]) if vq.sum()  > 0 else set()
    to  = set(np.argpartition(vo, -k)[-k:]) if vo.sum()  > 0 else set()
    u   = tq | to
    return len(tq & to) / len(u) if u else 0.0

def _jaccard_block(df, vec, BATCH=300):
    res = {f"jaccard_q_{opt}": np.zeros(len(df), dtype=np.float32)
           for opt in ["A","B","C","D"]}
    for s in range(0, len(df), BATCH):
        e = min(s+BATCH, len(df)); b = df.iloc[s:e]
        qm = vec.transform(b["question_clean"]).toarray()
        for opt in ["A","B","C","D"]:
            om = vec.transform(b[f"{opt}_clean"]).toarray()
            for li in range(e-s):
                res[f"jaccard_q_{opt}"][s+li] = _jaccard_topk(qm[li], om[li])
        if s % (BATCH*30) == 0:
            log.info(f"    Jaccard: {e:,}/{len(df):,} ...")
    return pd.DataFrame(res, index=df.index)

# ── Sentence cosine feature ───────────────────────────────────────────────────
def _sentence_cos_block(df, vec):
    scores = np.zeros(len(df), dtype=np.float32)
    for i in range(len(df)):
        sents = [s.strip() for s in re.split(r"[.\n?]", df["article_clean"].iloc[i])
                 if len(s.strip()) > 10]
        if not sents:
            continue
        qv = vec.transform([df["question_clean"].iloc[i]])
        sm = vec.transform(sents)
        cv = np.array(sm.dot(qv.T).todense()).ravel()
        scores[i] = float(cv.max()) if len(cv) > 0 else 0.0
        if i % 10_000 == 0 and i > 0:
            log.info(f"    Sentence cosine: {i:,}/{len(df):,} ...")
    return pd.Series(scores, name="max_sentence_cos", index=df.index)

# ── Main preprocessing function ───────────────────────────────────────────────
def run_preprocessing(max_vocab: int = DEFAULT_MAX_VOCAB) -> dict:
    log.info("\n" + "="*68)
    log.info("PHASE 1 — PREPROCESSING")
    log.info("="*68)

    # Load
    log.info("\n[Step 0] Loading RACE dataset ...")
    csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {RAW_DATA_PATH}")
    df = pd.read_csv(csv_files[0])
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]
    if unnamed: df.drop(columns=unnamed, inplace=True)
    log.info(f"  Loaded {len(df):,} rows from {csv_files[0].name}")

    train_raw, temp  = train_test_split(df,    test_size=0.20, random_state=RANDOM_SEED)
    dev_raw,  test_raw = train_test_split(temp, test_size=0.50, random_state=RANDOM_SEED)
    log.info(f"  Split -> train:{len(train_raw):,}  dev:{len(dev_raw):,}  test:{len(test_raw):,}")

    # Clean
    log.info("\n[Step 1] Cleaning ...")
    train_df = clean_dataframe(train_raw)
    dev_df   = clean_dataframe(dev_raw)
    test_df  = clean_dataframe(test_raw)

    # Combined text
    log.info("\n[Step 2] Combined text (article x2) ...")
    train_df = build_combined_text(train_df)
    dev_df   = build_combined_text(dev_df)
    test_df  = build_combined_text(test_df)

    # Lexical features
    log.info("\n[Step 3] Lexical features ...")
    train_df = build_lexical_features(train_df)
    dev_df   = build_lexical_features(dev_df)
    test_df  = build_lexical_features(test_df)

    # OHE
    log.info("\n[Step 4] OHE (fit on train only) ...")
    ohe_vec   = fit_ohe_vectorizer(train_df["combined_text"], max_vocab)
    train_ohe = ohe_vec.transform(train_df["combined_text"])
    dev_ohe   = ohe_vec.transform(dev_df["combined_text"])
    test_ohe  = ohe_vec.transform(test_df["combined_text"])

    # TF-IDF
    log.info("\n[Step 5] TF-IDF (fit on train only) ...")
    tfidf_vec   = fit_tfidf_vectorizer(train_df["combined_text"], max_vocab)
    train_tfidf = tfidf_vec.transform(train_df["combined_text"])
    dev_tfidf   = tfidf_vec.transform(dev_df["combined_text"])
    test_tfidf  = tfidf_vec.transform(test_df["combined_text"])
    inspect_vocab(tfidf_vec, train_tfidf)

    # OHE cosine
    log.info("\n[Step 6] OHE cosine similarity ...")
    for df_, split in [(train_df,"train"),(dev_df,"dev"),(test_df,"test")]:
        log.info(f"  -> {split}")
        block = _cosine_block(df_, ohe_vec, "ohe")
        for c in block.columns: df_[c] = block[c].values

    # TF-IDF cosine
    log.info("\n[Step 7] TF-IDF cosine similarity ...")
    for df_, split in [(train_df,"train"),(dev_df,"dev"),(test_df,"test")]:
        log.info(f"  -> {split}")
        block = _cosine_block(df_, tfidf_vec, "tfidf")
        for c in block.columns: df_[c] = block[c].values

    # Jaccard
    log.info("\n[Step 8] Jaccard TF-IDF overlap ...")
    for df_, split in [(train_df,"train"),(dev_df,"dev"),(test_df,"test")]:
        log.info(f"  -> {split}")
        block = _jaccard_block(df_, tfidf_vec)
        for c in block.columns: df_[c] = block[c].values

    # Sentence cosine
    log.info("\n[Step 9] Per-sentence max cosine ...")
    for df_, split in [(train_df,"train"),(dev_df,"dev"),(test_df,"test")]:
        log.info(f"  -> {split}")
        df_["max_sentence_cos"] = _sentence_cos_block(df_, tfidf_vec).values

    # Label encoding
    log.info("\n[Step 10] Label encoding ...")
    le = LabelEncoder()
    le.fit(["A","B","C","D"])
    for df_ in [train_df, dev_df, test_df]:
        df_["label"] = le.transform(df_["answer"])
    log.info(f"  Label dist (train):\n{train_df['label'].value_counts().sort_index().to_string()}")

    # Save
    log.info("\n[Step 11] Saving artefacts ...")
    for name, df_, ohe_, tfidf_ in [
        ("train", train_df, train_ohe, train_tfidf),
        ("dev",   dev_df,   dev_ohe,   dev_tfidf),
        ("test",  test_df,  test_ohe,  test_tfidf),
    ]:
        df_.to_csv(PROCESSED_PATH / f"{name}_clean.csv", index=False)
        sp.save_npz(str(PROCESSED_PATH / f"{name}_ohe.npz"),   ohe_)
        sp.save_npz(str(PROCESSED_PATH / f"{name}_tfidf.npz"), tfidf_)

    joblib.dump(ohe_vec,   MODELS_TRAD / "ohe_vectorizer.joblib")
    joblib.dump(tfidf_vec, MODELS_TRAD / "tfidf_vectorizer.joblib")
    joblib.dump(le,        MODELS_TRAD / "label_encoder.joblib")

    config = {
        "ohe_vocab":   len(ohe_vec.vocabulary_),
        "tfidf_vocab": len(tfidf_vec.vocabulary_),
        "train_rows":  len(train_df),
        "dev_rows":    len(dev_df),
        "test_rows":   len(test_df),
        "dense_cols":  DENSE_COLS,
        "saved_at":    datetime.now().isoformat(timespec="seconds"),
    }
    with open(PROCESSED_PATH / "preprocessing_config.json", "w") as f:
        json.dump(config, f, indent=2)

    log.info(f"\n  Preprocessing complete. All files in {PROCESSED_PATH}")
    return {
        "train_df":train_df,"dev_df":dev_df,"test_df":test_df,
        "train_ohe":train_ohe,"dev_ohe":dev_ohe,"test_ohe":test_ohe,
        "train_tfidf":train_tfidf,"dev_tfidf":dev_tfidf,"test_tfidf":test_tfidf,
        "ohe_vec":ohe_vec,"tfidf_vec":tfidf_vec,"le":le,
    }


/tmp/ipykernel_57/3780639878.py:119: UserWarning: PyTorch / transformers not found. BERT phase will be skipped.
  warnings.warn("PyTorch / transformers not found. BERT phase will be skipped.")
22:28:14 [INFO] 
####################################################################
22:28:14 [INFO]   MODEL A — FULL PIPELINE  (Kaggle)
22:28:14 [INFO] ####################################################################
22:28:14 [INFO] 
22:28:14 [INFO] PHASE 1 — PREPROCESSING
22:28:14 [INFO] ====================================================================
22:28:14 [INFO] 
[Step 0] Loading RACE dataset ...
22:28:17 [INFO]   Loaded 87,866 rows from dev.csv
22:28:17 [INFO]   Split -> train:70,292  dev:8,787  test:8,787
22:28:17 [INFO] 
[Step 1] Cleaning ...
22:28:17 [INFO]   Cleaning text columns ...
22:28:26 [INFO]   Rows after cleaning: 70,270  (removed 3 dupes)
22:28:26 [INFO]   Cleaning text columns ...
22:28:27 [INFO]   Rows after cleaning: 8,786  (removed 0 dupes)
22:28:27 [INFO]   Clea

In [ ]:


# #############################################################################
#
#  PHASE 2 — CLASSICAL ML MODELS
#
# #############################################################################

# ── Data loader ───────────────────────────────────────────────────────────────
def load_split(split: str, quick: bool = False):
    df      = pd.read_csv(PROCESSED_PATH / f"{split}_clean.csv")
    X_ohe   = sp.load_npz(str(PROCESSED_PATH / f"{split}_ohe.npz"))
    X_tfidf = sp.load_npz(str(PROCESSED_PATH / f"{split}_tfidf.npz"))
    dc      = [c for c in DENSE_COLS if c in df.columns]
    X_dense = df[dc].fillna(0).values.astype(np.float32)
    y       = df["label"].values.astype(int)
    if quick:
        n   = min(20_000, len(df))
        idx = np.random.RandomState(RANDOM_SEED).choice(len(df), n, replace=False)
        df, X_ohe, X_tfidf, X_dense, y = (
            df.iloc[idx].reset_index(drop=True),
            X_ohe[idx], X_tfidf[idx], X_dense[idx], y[idx])
        log.info(f"  [QUICK] {split}: {n:,} rows")
    else:
        log.info(f"  {split}: {len(df):,} rows | OHE:{X_ohe.shape} | TFIDF:{X_tfidf.shape} | dense:{X_dense.shape}")
    return X_ohe, X_tfidf, X_dense, y, df

# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(y_true, y_pred, label="") -> dict:
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm  = confusion_matrix(y_true, y_pred)
    log.info(f"\n  [{label}]")
    log.info(f"    Accuracy    : {acc:.4f}  ({acc*100:.2f}%)")
    log.info(f"    Macro F1    : {f1:.4f}")
    log.info(f"    Exact Match : {acc:.4f}")
    log.info(f"    Confusion Matrix:\n{cm}")
    return {"model": label, "accuracy": acc, "macro_f1": f1, "exact_match": acc}

def clustering_purity(y_true, labels):
    total = len(y_true); correct = 0
    for c in np.unique(labels):
        mask = labels == c
        if mask.sum(): correct += Counter(y_true[mask]).most_common(1)[0][1]
    return correct / total

def save_model(model, name: str):
    p = MODELS_TRAD / f"{name}.joblib"
    joblib.dump(model, p)
    log.info(f"  Saved: {p}")

# ── Per-option feature builder ────────────────────────────────────────────────
def build_per_option_features(df: pd.DataFrame, tfidf_vec, ohe_vec):
    """
    THE CRITICAL FIX for LR / SVM / NB low accuracy.

    Root cause of ~20% accuracy:
        The original combined_text = article + article + question + A + B + C + D
        All four options are always present in every sample.
        The model has no way to learn which option is correct — it sees the same
        4 options in every row regardless of label. This produces near-random output.

    Fix — 4-option comparative format:
        For each (article, question) row, build 4 separate TF-IDF/OHE vectors:
            combined_A = article x2 + question + option_A
            combined_B = article x2 + question + option_B
            combined_C = article x2 + question + option_C
            combined_D = article x2 + question + option_D

        Then horizontally stack all 4 option vectors into a single feature row:
            X_row = [tfidf(combined_A) | tfidf(combined_B) |
                     tfidf(combined_C) | tfidf(combined_D)]

        Now the model sees ALL four options side-by-side and can learn
        positional + content patterns that identify the correct option.
        This is the standard IR/QA approach for multiple-choice tasks.

    Returns:
        X_tfidf_4opt : sparse (N, vocab*4) — for LR and SVM
        X_ohe_4opt   : sparse (N, vocab*4) — for NB
    """
    log.info("  Building per-option comparative feature vectors ...")
    BATCH = 500
    n = len(df)

    tfidf_parts = {opt: [] for opt in ["A","B","C","D"]}
    ohe_parts   = {opt: [] for opt in ["A","B","C","D"]}

    for s in range(0, n, BATCH):
        e = min(s + BATCH, n)
        b = df.iloc[s:e]
        for opt in ["A","B","C","D"]:
            combined = (
                b["article_clean"].fillna("").astype(str) + " " +
                b["article_clean"].fillna("").astype(str) + " " +
                b["question_clean"].fillna("").astype(str) + " " +
                b[f"{opt}_clean"].fillna("").astype(str)
            ).fillna("").astype(str)          # final guard on the whole series
            tfidf_parts[opt].append(tfidf_vec.transform(combined))
            ohe_parts[opt].append(ohe_vec.transform(combined))


    # Stack all batches per option, then hstack options side-by-side
    X_tfidf_4opt = sp.hstack([
        sp.vstack(tfidf_parts["A"]),
        sp.vstack(tfidf_parts["B"]),
        sp.vstack(tfidf_parts["C"]),
        sp.vstack(tfidf_parts["D"]),
    ], format="csr")

    X_ohe_4opt = sp.hstack([
        sp.vstack(ohe_parts["A"]),
        sp.vstack(ohe_parts["B"]),
        sp.vstack(ohe_parts["C"]),
        sp.vstack(ohe_parts["D"]),
    ], format="csr")

    log.info(f"  Per-option TF-IDF matrix: {X_tfidf_4opt.shape}")
    log.info(f"  Per-option OHE matrix   : {X_ohe_4opt.shape}")
    return X_tfidf_4opt, X_ohe_4opt


# ── S1: Logistic Regression ───────────────────────────────────────────────────
def train_lr(Xtr, ytr, Xdv, ydv):
    """
    Input: per-option comparative TF-IDF matrix (N, vocab*4).
    Model can now compare all four options and learn which one is correct.
    """
    log.info("\n[S1] Logistic Regression ...")
    m = LogisticRegression(C=1.0, solver="saga", max_iter=1000,
                           multi_class="multinomial", class_weight="balanced",
                           random_state=RANDOM_SEED, n_jobs=-1)
    m.fit(Xtr, ytr)
    r = evaluate(ydv, m.predict(Xdv), "Logistic Regression (TF-IDF per-option)")
    save_model(m, "lr_tfidf")
    return r, m

# ── S2: SVM ───────────────────────────────────────────────────────────────────
def train_svm(Xtr, ytr, Xdv, ydv):
    """
    Input: per-option comparative TF-IDF matrix (N, vocab*4).
    LinearSVC handles high-dimensional sparse data efficiently.
    """
    log.info("\n[S2] SVM (LinearSVC + Calibrated) ...")
    base = LinearSVC(C=0.1, max_iter=3000, class_weight="balanced",
                     random_state=RANDOM_SEED)
    m = CalibratedClassifierCV(base, cv=3, method="sigmoid")
    m.fit(Xtr, ytr)
    r = evaluate(ydv, m.predict(Xdv), "SVM (LinearSVC + Calibrated)")
    save_model(m, "svm_calibrated")
    return r, m

# ── S3: Naive Bayes ───────────────────────────────────────────────────────────
def train_nb(Xtr_ohe, ytr, Xdv_ohe, ydv):
    """
    Input: per-option comparative OHE matrix (N, vocab*4).
    ComplementNB on binary OHE input — non-negative, correct for MultinomialNB family.
    """
    log.info("\n[S3] Naive Bayes (ComplementNB) ...")
    m = ComplementNB(alpha=0.3)
    m.fit(Xtr_ohe, ytr)
    r = evaluate(ydv, m.predict(Xdv_ohe), "Naive Bayes (ComplementNB + OHE)")
    save_model(m, "naive_bayes")
    return r, m

# ── S4: Random Forest ─────────────────────────────────────────────────────────
def train_rf(Xtr_d, ytr, Xdv_d, ydv):
    log.info("\n[S4] Random Forest (dense features) ...")
    m = RandomForestClassifier(n_estimators=500, max_depth=20,
                               min_samples_leaf=5, max_features="sqrt",
                               class_weight="balanced",
                               random_state=RANDOM_SEED, n_jobs=-1)
    m.fit(Xtr_d, ytr)
    r = evaluate(ydv, m.predict(Xdv_d), "Random Forest (dense features)")
    imp = m.feature_importances_
    top = imp.argsort()[::-1][:10]
    log.info("  Top-10 RF feature importances:")
    for i, idx in enumerate(top, 1):
        name = DENSE_COLS[idx] if idx < len(DENSE_COLS) else f"feat_{idx}"
        log.info(f"    {i:2d}. {name:30s} {imp[idx]:.4f}")
    save_model(m, "random_forest")
    return r, m

# ── S5: XGBoost ───────────────────────────────────────────────────────────────
def train_xgb(Xtr_d, ytr, Xdv_d, ydv):
    if not XGBOOST_AVAILABLE:
        log.warning("[S5] XGBoost not available — skipping.")
        return {"model":"XGBoost","accuracy":None,"macro_f1":None,"exact_match":None}, None
    log.info("\n[S5] XGBoost (dense features) ...")
    m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      eval_metric="mlogloss", use_label_encoder=False,
                      random_state=RANDOM_SEED, n_jobs=-1)
    m.fit(Xtr_d, ytr, eval_set=[(Xdv_d, ydv)], verbose=False)
    r = evaluate(ydv, m.predict(Xdv_d), "XGBoost (dense features)")
    save_model(m, "xgboost")
    return r, m

# ── U1: K-Means ───────────────────────────────────────────────────────────────
def train_kmeans(Xtr_tfidf, ytr, Xdv_tfidf, ydv):
    log.info("\n[U1] K-Means ...")
    svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_SEED)
    Xtr_lsa = normalize(svd.fit_transform(Xtr_tfidf))
    Xdv_lsa = normalize(svd.transform(Xdv_tfidf))
    log.info(f"  LSA explained variance: {svd.explained_variance_ratio_.sum():.3f}")
    m = MiniBatchKMeans(n_clusters=KMEANS_K, init="k-means++", n_init=10,
                        max_iter=300, batch_size=4096, random_state=RANDOM_SEED)
    m.fit(Xtr_lsa)
    cl  = m.predict(Xdv_lsa)
    pur = clustering_purity(ydv, cl)
    sil = silhouette_score(Xdv_lsa[:5000], cl[:5000], metric="cosine",
                           sample_size=3000, random_state=RANDOM_SEED)
    log.info(f"\n  [K-Means]  Purity:{pur:.4f}  Silhouette:{sil:.4f}  Cluster sizes:{Counter(cl)}")
    save_model(m,   "kmeans")
    save_model(svd, "kmeans_svd")
    return {"model":"K-Means","purity":pur,"silhouette":sil}, m

# ── U2: GMM ───────────────────────────────────────────────────────────────────
def train_gmm(Xtr_d, ytr, Xdv_d, ydv):
    log.info("\n[U2] Gaussian Mixture Model ...")
    m = GaussianMixture(n_components=GMM_K, covariance_type="full",
                        n_init=5, max_iter=200, random_state=RANDOM_SEED)
    m.fit(Xtr_d)
    cl  = m.predict(Xdv_d)
    pur = clustering_purity(ydv, cl)
    idx = np.random.RandomState(RANDOM_SEED).choice(
        len(Xdv_d), min(3000, len(Xdv_d)), replace=False)
    sil = silhouette_score(Xdv_d[idx], cl[idx])
    log.info(f"\n  [GMM]  Purity:{pur:.4f}  Silhouette:{sil:.4f}  Cluster sizes:{Counter(cl)}")
    save_model(m, "gmm")
    return {"model":"GMM","purity":pur,"silhouette":sil}, m

# ── U3: Label Propagation ─────────────────────────────────────────────────────
LP_MAX_ROWS = 5_000   # MEMORY FIX: RBF LP builds N×N affinity matrix.
                       # 70k rows = 70k×70k = ~39 GB RAM → kernel crash.
                       # 5k rows = 5k×5k = ~200 MB → safe on Kaggle (16 GB RAM).

def train_lp(Xtr_d, ytr, Xdv_d, ydv):
    """
    Label Propagation with mandatory row cap to prevent OOM crash.

    Why it crashes without the cap:
        LabelPropagation(kernel='rbf') internally builds a dense affinity
        matrix of shape (n_samples, n_samples).
        70,270 × 70,270 × 8 bytes = ~39 GB  →  kernel restart.

    Fix: subsample to LP_MAX_ROWS (5,000) rows — stratified so each class
    is represented. Predict on full dev set using the trained LP model.
    """
    log.info("\n[U3] Label Propagation (semi-supervised) ...")
    log.info(f"  Subsampling to {LP_MAX_ROWS:,} rows to prevent OOM (N×N affinity matrix).")

    rng = np.random.RandomState(RANDOM_SEED)

    # Stratified subsample of training rows
    sub_idx = []
    per_class = LP_MAX_ROWS // N_CLASSES
    for cls in range(N_CLASSES):
        ci = np.where(ytr == cls)[0]
        sub_idx.extend(rng.choice(ci, min(per_class, len(ci)), replace=False).tolist())
    sub_idx = np.array(sub_idx)

    Xtr_sub = Xtr_d[sub_idx]
    ytr_sub = ytr[sub_idx]

    # Mark only LABEL_PROP_FRAC of the subsample as labeled
    n_labeled = max(int(LABEL_PROP_FRAC * len(ytr_sub)), N_CLASSES * 5)
    y_partial = np.full_like(ytr_sub, fill_value=-1)
    for cls in range(N_CLASSES):
        ci     = np.where(ytr_sub == cls)[0]
        n      = n_labeled // N_CLASSES
        chosen = rng.choice(ci, min(n, len(ci)), replace=False)
        y_partial[chosen] = cls

    log.info(f"  Subsample: {len(ytr_sub):,} rows | "
             f"Labeled: {(y_partial!=-1).sum():,} | Unlabeled: {(y_partial==-1).sum():,}")

    # knn kernel is O(n * k) memory — far safer than rbf for larger n
    m = LabelPropagation(kernel="knn", n_neighbors=7, max_iter=1000, tol=1e-3, n_jobs=-1)
    m.fit(Xtr_sub, y_partial)

    r = evaluate(ydv, m.predict(Xdv_d),
                 f"Label Propagation ({LABEL_PROP_FRAC*100:.0f}% labeled, {LP_MAX_ROWS:,} rows)")
    save_model(m, "label_propagation")
    return r, m

# ── Ensembles ─────────────────────────────────────────────────────────────────
def train_ensembles(models, Xdv_tfidf_4opt, Xdv_ohe_4opt, Xdv_d, ydv):
    lr_m, svm_m, nb_m, rf_m, xgb_m = (
        models.get("lr"), models.get("svm"), models.get("nb"),
        models.get("rf"), models.get("xgb"))
    results = []

    # E1: Soft voting — LR + SVM use per-option TF-IDF; NB uses per-option OHE
    log.info("\n[E1] Soft Voting (LR + SVM + NB) ...")
    p_lr  = lr_m.predict_proba(Xdv_tfidf_4opt)
    p_svm = svm_m.predict_proba(Xdv_tfidf_4opt)
    p_nb  = nb_m.predict_proba(Xdv_ohe_4opt)
    y_soft = ((p_lr + p_svm + p_nb) / 3.0).argmax(axis=1)
    results.append(evaluate(ydv, y_soft, "Soft Voting (LR + SVM + NB)"))

    # E2: Hard voting
    log.info("\n[E2] Hard Voting (all models) ...")
    votes = [lr_m.predict(Xdv_tfidf_4opt), svm_m.predict(Xdv_tfidf_4opt),
             nb_m.predict(Xdv_ohe_4opt),   rf_m.predict(Xdv_d)]
    if xgb_m: votes.append(xgb_m.predict(Xdv_d))
    va     = np.stack(votes, axis=1)
    y_hard = np.array([Counter(row).most_common(1)[0][0] for row in va])
    results.append(evaluate(ydv, y_hard, "Hard Voting (all models)"))

    return results

def train_stacking(models, Xtr_tfidf_4opt, Xtr_ohe_4opt, Xtr_d, ytr,
                              Xdv_tfidf_4opt, Xdv_ohe_4opt, Xdv_d, ydv):
    log.info("\n[E3] Stacking (OOF meta-LR) ...")
    lr_m, svm_m, nb_m, rf_m, xgb_m = (
        models.get("lr"), models.get("svm"), models.get("nb"),
        models.get("rf"), models.get("xgb"))
    n_base = 4 + (1 if xgb_m else 0)
    oof    = np.zeros((len(ytr), N_CLASSES * n_base), dtype=np.float32)
    skf    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    for fold, (tri, vli) in enumerate(skf.split(Xtr_d, ytr)):
        log.info(f"  Stacking fold {fold+1}/{CV_FOLDS} ...")
        # LR — per-option TF-IDF
        fl = LogisticRegression(C=1.0, solver="saga", max_iter=500,
                                multi_class="multinomial", class_weight="balanced",
                                random_state=RANDOM_SEED, n_jobs=-1)
        fl.fit(Xtr_tfidf_4opt[tri], ytr[tri])
        oof[vli, 0:4]  = fl.predict_proba(Xtr_tfidf_4opt[vli])
        # SVM — per-option TF-IDF
        fs = CalibratedClassifierCV(
            LinearSVC(C=0.1, max_iter=3000, class_weight="balanced",
                      random_state=RANDOM_SEED), cv=2, method="sigmoid")
        fs.fit(Xtr_tfidf_4opt[tri], ytr[tri])
        oof[vli, 4:8]  = fs.predict_proba(Xtr_tfidf_4opt[vli])
        # NB — per-option OHE
        fn = ComplementNB(alpha=0.3)
        fn.fit(Xtr_ohe_4opt[tri], ytr[tri])
        oof[vli, 8:12] = fn.predict_proba(Xtr_ohe_4opt[vli])
        # RF — dense
        fr = RandomForestClassifier(n_estimators=100, max_depth=15,
                                    class_weight="balanced",
                                    random_state=RANDOM_SEED, n_jobs=-1)
        fr.fit(Xtr_d[tri], ytr[tri])
        oof[vli, 12:16] = fr.predict_proba(Xtr_d[vli])
        # XGB — dense
        if xgb_m:
            fx = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.05,
                               subsample=0.8, eval_metric="mlogloss",
                               use_label_encoder=False,
                               random_state=RANDOM_SEED, n_jobs=-1)
            fx.fit(Xtr_d[tri], ytr[tri])
            oof[vli, 16:20] = fx.predict_proba(Xtr_d[vli])

    meta_lr = LogisticRegression(C=1.0, max_iter=500, random_state=RANDOM_SEED)
    meta_lr.fit(oof, ytr)

    dev_meta = np.hstack([
        lr_m.predict_proba(Xdv_tfidf_4opt),
        svm_m.predict_proba(Xdv_tfidf_4opt),
        nb_m.predict_proba(Xdv_ohe_4opt),
        rf_m.predict_proba(Xdv_d),
    ])
    if xgb_m: dev_meta = np.hstack([dev_meta, xgb_m.predict_proba(Xdv_d)])
    r = evaluate(ydv, meta_lr.predict(dev_meta), "Stacking (meta-LR)")
    save_model(meta_lr, "stacking_meta_lr")
    return r, meta_lr

def save_results(sup_results, uns_results):
    RESULTS_PATH.mkdir(parents=True, exist_ok=True)
    sup_df = pd.DataFrame([r for r in sup_results if r.get("accuracy") is not None])
    uns_df = pd.DataFrame(uns_results)
    if not sup_df.empty:
        sup_df.sort_values("macro_f1", ascending=False).to_csv(
            RESULTS_PATH / "model_a_supervised_results.csv", index=False)
        log.info(f"\n  Best supervised model: {sup_df.sort_values('macro_f1',ascending=False).iloc[0]['model']}")
        log.info(f"    Accuracy: {sup_df.sort_values('macro_f1',ascending=False).iloc[0]['accuracy']:.4f}")
        log.info(f"    Macro F1: {sup_df.sort_values('macro_f1',ascending=False).iloc[0]['macro_f1']:.4f}")
    if not uns_df.empty:
        uns_df.to_csv(RESULTS_PATH / "model_a_unsupervised_results.csv", index=False)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pd.concat([sup_df, uns_df], ignore_index=True).to_csv(
        RESULTS_PATH / f"model_a_results_{ts}.csv", index=False)

# ── Main classical ML function ────────────────────────────────────────────────
def run_classical_models(quick: bool = False) -> dict:
    log.info("\n" + "="*68)
    log.info("PHASE 2 — CLASSICAL ML MODELS")
    log.info("="*68)

    log.info("\n[Load] Train ...")
    Xtr_ohe, Xtr_tfidf, Xtr_d, ytr, train_df = load_split("train", quick)
    log.info("\n[Load] Dev ...")
    Xdv_ohe, Xdv_tfidf, Xdv_d, ydv, dev_df   = load_split("dev",   quick)

    # Load fitted vectorizers (saved during preprocessing)
    log.info("\n[Load] Vectorizers ...")
    tfidf_vec = joblib.load(MODELS_TRAD / "tfidf_vectorizer.joblib")
    ohe_vec   = joblib.load(MODELS_TRAD / "ohe_vectorizer.joblib")

    # Build per-option comparative matrices for LR / SVM / NB
    log.info("\n[Build] Per-option comparative feature matrices ...")
    log.info("  Train ...")
    Xtr_tfidf_4opt, Xtr_ohe_4opt = build_per_option_features(train_df, tfidf_vec, ohe_vec)
    log.info("  Dev ...")
    Xdv_tfidf_4opt, Xdv_ohe_4opt = build_per_option_features(dev_df,   tfidf_vec, ohe_vec)

    # Free raw combined matrices — no longer needed
    del Xtr_tfidf, Xtr_ohe, Xdv_tfidf, Xdv_ohe
    gc.collect()

    sup_r, uns_r, mods = [], [], {}

    # Supervised — LR/SVM/NB use per-option matrices; RF/XGB use dense
    r, m = train_lr(Xtr_tfidf_4opt, ytr, Xdv_tfidf_4opt, ydv);  sup_r.append(r); mods["lr"]  = m
    r, m = train_svm(Xtr_tfidf_4opt, ytr, Xdv_tfidf_4opt, ydv); sup_r.append(r); mods["svm"] = m
    r, m = train_nb(Xtr_ohe_4opt, ytr, Xdv_ohe_4opt, ydv);      sup_r.append(r); mods["nb"]  = m
    r, m = train_rf(Xtr_d, ytr, Xdv_d, ydv);                    sup_r.append(r); mods["rf"]  = m
    r, m = train_xgb(Xtr_d, ytr, Xdv_d, ydv);                   sup_r.append(r); mods["xgb"] = m

    # Unsupervised (use original TF-IDF from disk for K-Means)
    Xtr_tfidf_raw = sp.load_npz(str(PROCESSED_PATH / "train_tfidf.npz"))
    Xdv_tfidf_raw = sp.load_npz(str(PROCESSED_PATH / "dev_tfidf.npz"))
    r, m = train_kmeans(Xtr_tfidf_raw, ytr, Xdv_tfidf_raw, ydv); uns_r.append(r); mods["kmeans"] = m
    del Xtr_tfidf_raw, Xdv_tfidf_raw; gc.collect()

    r, m = train_gmm(Xtr_d, ytr, Xdv_d, ydv);  uns_r.append(r); mods["gmm"] = m
    r, m = train_lp(Xtr_d,  ytr, Xdv_d, ydv);  sup_r.append(r); mods["lp"]  = m

    # Ensembles
    ens_r = train_ensembles(mods, Xdv_tfidf_4opt, Xdv_ohe_4opt, Xdv_d, ydv)
    sup_r.extend(ens_r)

    # Stacking
    if not quick:
        r, m = train_stacking(mods,
                              Xtr_tfidf_4opt, Xtr_ohe_4opt, Xtr_d, ytr,
                              Xdv_tfidf_4opt, Xdv_ohe_4opt, Xdv_d, ydv)
        sup_r.append(r); mods["stacking"] = m
    else:
        log.info("  [E3] Stacking skipped in quick mode.")

    save_results(sup_r, uns_r)

    # Test evaluation on best model
    log.info("\n[Final Test] Evaluating best model on test set ...")
    best = max([r for r in sup_r if r.get("accuracy")], key=lambda r: r["macro_f1"])
    log.info(f"  Best: {best['model']}")
    Xts_ohe_raw, Xts_tfidf_raw, Xts_d, yts, test_df = load_split("test", quick)
    log.info("  Building test per-option matrices ...")
    Xts_tfidf_4opt, Xts_ohe_4opt = build_per_option_features(test_df, tfidf_vec, ohe_vec)
    del Xts_ohe_raw, Xts_tfidf_raw; gc.collect()

    name_map = {
        "Logistic Regression (TF-IDF per-option)": ("lr",       Xts_tfidf_4opt),
        "SVM (LinearSVC + Calibrated)":            ("svm",      Xts_tfidf_4opt),
        "Naive Bayes (ComplementNB + OHE)":        ("nb",       Xts_ohe_4opt),
        "Random Forest (dense features)":          ("rf",       Xts_d),
        "XGBoost (dense features)":                ("xgb",      Xts_d),
        "Label Propagation (5% labeled, 5,000 rows)": ("lp",    Xts_d),
        "Stacking (meta-LR)":                      ("stacking", None),
    }
    # Handle slight label name variation for LP
    for k in list(name_map.keys()):
        if k.startswith("Label Propagation"):
            name_map[best["model"]] = name_map[k]  # match whatever name was set
            break

    bname = best["model"]
    yp    = None
    if bname in name_map:
        mk, Xts = name_map[bname]
        if mk == "stacking" and "stacking" in mods:
            tm = np.hstack([mods["lr"].predict_proba(Xts_tfidf_4opt),
                            mods["svm"].predict_proba(Xts_tfidf_4opt),
                            mods["nb"].predict_proba(Xts_ohe_4opt),
                            mods["rf"].predict_proba(Xts_d)])
            if mods.get("xgb"): tm = np.hstack([tm, mods["xgb"].predict_proba(Xts_d)])
            yp = mods["stacking"].predict(tm)
        elif mk in mods and mods[mk] is not None:
            yp = mods[mk].predict(Xts)
    if yp is not None:
        tr = evaluate(yts, yp, f"TEST — {bname}")
        pd.DataFrame([tr]).to_csv(RESULTS_PATH / "model_a_test_result.csv", index=False)

    log.info("\nPhase 2 complete.")
    return mods


# #############################################################################
#
#  PHASE 3 — BERT FINE-TUNING
#
# #############################################################################

class RACEDataset:
    """PyTorch Dataset — works only when TORCH_AVAILABLE is True."""
    def __init__(self, df, tokenizer, max_length=BERT_MAX_LENGTH, quick=False):
        if quick: df = df.head(2000).reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.articles   = df["article"].fillna("").tolist()
        self.questions  = df["question"].fillna("").tolist()
        self.opts       = {opt: df[opt].fillna("").tolist() for opt in ["A","B","C","D"]}
        self.labels     = df["label"].values.astype(int)
        self._idx_map   = {0:"A",1:"B",2:"C",3:"D"}
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        import torch
        label  = int(self.labels[idx])
        option = self.opts[self._idx_map[label]][idx]
        enc = self.tokenizer(
            self.questions[idx] + " " + option,
            self.articles[idx],
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc.get("token_type_ids",
                              torch.zeros(self.max_length, dtype=torch.long)).squeeze(0),
            "labels":         torch.tensor(label, dtype=torch.long),
        }


def run_bert(epochs: int = 3, batch_size: int = 16, grad_accum: int = 2,
             lr: float = 2e-5, weight_decay: float = 0.01,
             max_length: int = BERT_MAX_LENGTH, quick: bool = False):

    if not TORCH_AVAILABLE:
        log.error("PyTorch / transformers not installed. Skipping BERT phase.")
        log.error("Install: !pip install transformers torch -q")
        return

    import torch
    from torch import nn
    from torch.utils.data import DataLoader
    from torch.cuda.amp import autocast, GradScaler

    log.info("\n" + "="*68)
    log.info("PHASE 3 — BERT FINE-TUNING")
    log.info(f"  Model      : {BERT_MODEL_NAME}")
    log.info(f"  Epochs     : {epochs}")
    log.info(f"  Batch      : {batch_size} x accum {grad_accum} = eff. {batch_size*grad_accum}")
    log.info(f"  LR         : {lr}")
    log.info(f"  Quick mode : {quick}")
    log.info("="*68)

    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_fp16 = torch.cuda.is_available()
    scaler   = GradScaler() if use_fp16 else None
    log.info(f"  Device: {device}  |  FP16: {use_fp16}")

    MODELS_BERT.mkdir(parents=True, exist_ok=True)

    # Load tokenizer & data
    log.info("\n[BERT-Data] Loading tokenizer ...")
    tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL_NAME)

    log.info("[BERT-Data] Loading splits ...")
    train_df = pd.read_csv(PROCESSED_PATH / "train_clean.csv")
    dev_df   = pd.read_csv(PROCESSED_PATH / "dev_clean.csv")
    test_df  = pd.read_csv(PROCESSED_PATH / "test_clean.csv")

    train_ds = RACEDataset(train_df, tokenizer, max_length, quick)
    dev_ds   = RACEDataset(dev_df,   tokenizer, max_length, quick)
    test_ds  = RACEDataset(test_df,  tokenizer, max_length, quick)
    log.info(f"  Train:{len(train_ds):,}  Dev:{len(dev_ds):,}  Test:{len(test_ds):,}")

    nw = 2 if not quick else 0
    train_loader = DataLoader(train_ds, batch_size=batch_size,    shuffle=True,
                              num_workers=nw, pin_memory=torch.cuda.is_available())
    dev_loader   = DataLoader(dev_ds,   batch_size=batch_size*2,  shuffle=False,
                              num_workers=nw, pin_memory=torch.cuda.is_available())
    test_loader  = DataLoader(test_ds,  batch_size=batch_size*2,  shuffle=False,
                              num_workers=nw, pin_memory=torch.cuda.is_available())

    # Model
    log.info(f"\n[BERT-Model] Loading {BERT_MODEL_NAME} ...")
    model = BertForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME, num_labels=N_CLASSES,
        hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1)
    model.to(device)
    tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"  Trainable parameters: {tp:,}")

    # Optimizer — no weight decay on bias / LayerNorm
    no_decay = ["bias","LayerNorm.weight"]
    opt_groups = [
        {"params":[p for n,p in model.named_parameters()
                   if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay},
        {"params":[p for n,p in model.named_parameters()
                   if     any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer   = AdamW(opt_groups, lr=lr, eps=1e-8)
    total_steps = (len(train_loader) // grad_accum) * epochs
    warmup_steps= int(WARMUP_RATIO * total_steps)
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    log.info(f"  Total steps:{total_steps:,}  Warmup:{warmup_steps:,}")

    # Evaluation helper
    def bert_evaluate(loader, split_name):
        model.eval(); all_p, all_l = [], []
        with torch.no_grad():
            for batch in loader:
                iids = batch["input_ids"].to(device)
                amsk = batch["attention_mask"].to(device)
                ttids= batch["token_type_ids"].to(device)
                lbls = batch["labels"]
                if use_fp16:
                    with autocast():
                        out = model(input_ids=iids, attention_mask=amsk, token_type_ids=ttids)
                else:
                    out = model(input_ids=iids, attention_mask=amsk, token_type_ids=ttids)
                preds = out.logits.argmax(dim=-1).cpu().numpy()
                all_p.extend(preds); all_l.extend(lbls.numpy())
        yt, yp = np.array(all_l), np.array(all_p)
        acc = accuracy_score(yt, yp)
        f1  = f1_score(yt, yp, average="macro", zero_division=0)
        log.info(f"  [{split_name.upper()}]  Accuracy:{acc:.4f}  Macro F1:{f1:.4f}  EM:{acc:.4f}")
        return {"accuracy": acc, "macro_f1": f1, "exact_match": acc}

    # Training loop
    best_acc = 0.0; tlog = []
    log.info("\n[BERT-Train]")
    for epoch in range(1, epochs+1):
        log.info(f"\n--- Epoch {epoch}/{epochs} ---")
        model.train(); total_loss = 0.0; nb = 0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            iids  = batch["input_ids"].to(device)
            amsk  = batch["attention_mask"].to(device)
            ttids = batch["token_type_ids"].to(device)
            lbls  = batch["labels"].to(device)
            if use_fp16:
                with autocast():
                    out  = model(input_ids=iids, attention_mask=amsk,
                                 token_type_ids=ttids, labels=lbls)
                    loss = out.loss / grad_accum
                scaler.scale(loss).backward()
            else:
                out  = model(input_ids=iids, attention_mask=amsk,
                             token_type_ids=ttids, labels=lbls)
                loss = out.loss / grad_accum
                loss.backward()
            total_loss += out.loss.item(); nb += 1
            if (step+1) % grad_accum == 0:
                if use_fp16:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer); scaler.update()
                else:
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step(); optimizer.zero_grad()
            if step % 200 == 0:
                log.info(f"  Ep{epoch} | Step {step}/{len(train_loader)} | "
                         f"Loss:{total_loss/max(nb,1):.4f}")
        avg_loss = total_loss / max(nb, 1)
        log.info(f"  Epoch {epoch} avg loss: {avg_loss:.4f}")
        dm = bert_evaluate(dev_loader, "dev")
        if dm["accuracy"] > best_acc:
            best_acc = dm["accuracy"]
            model.save_pretrained(MODELS_BERT / "best_model")
            tokenizer.save_pretrained(MODELS_BERT / "tokenizer")
            log.info(f"  New best dev accuracy: {best_acc:.4f} — saved.")
        tlog.append({"epoch":epoch,"train_loss":avg_loss,
                     "dev_accuracy":dm["accuracy"],"dev_macro_f1":dm["macro_f1"]})

    # Test evaluation on best model
    log.info("\n[BERT-Test] Loading best model ...")
    model = BertForSequenceClassification.from_pretrained(
        MODELS_BERT / "best_model", num_labels=N_CLASSES)
    model.to(device)
    tm = bert_evaluate(test_loader, "test")

    # Save log & config
    pd.DataFrame(tlog).to_csv(MODELS_BERT / "training_log.csv", index=False)
    cfg = {
        "model": BERT_MODEL_NAME, "epochs": epochs,
        "batch_size": batch_size, "grad_accum": grad_accum,
        "lr": lr, "weight_decay": weight_decay,
        "max_length": max_length,
        "best_dev_accuracy": best_acc,
        "test_accuracy": tm["accuracy"],
        "test_macro_f1": tm["macro_f1"],
        "test_exact_match": tm["exact_match"],
        "device": str(device), "fp16": use_fp16,
        "saved_at": datetime.now().isoformat(timespec="seconds"),
    }
    with open(MODELS_BERT / "bert_config.json", "w") as f:
        json.dump(cfg, f, indent=2)
    pd.DataFrame([{"model":"BERT (bert-base-uncased)", **tm}]).to_csv(
        RESULTS_PATH / "model_a_bert_test_result.csv", index=False)

    log.info("\n" + "="*68)
    log.info("BERT COMPLETE")
    log.info(f"  Best Dev Accuracy : {best_acc:.4f}")
    log.info(f"  Test Accuracy     : {tm['accuracy']:.4f}")
    log.info(f"  Test Macro F1     : {tm['macro_f1']:.4f}")
    log.info(f"  Model saved to    : {(MODELS_BERT / 'best_model').resolve()}")
    log.info("="*68)



In [ ]:

# #############################################################################
#
#  MASTER RUNNER
#
# #############################################################################
def main(skip_preprocessing=False, skip_classical=False, skip_bert=False,
         bert_epochs=3, bert_batch=16, quick=False, max_vocab=DEFAULT_MAX_VOCAB):

    log.info("\n" + "#"*68)
    log.info("  MODEL A — FULL PIPELINE  (Kaggle)")
    log.info("#"*68)

    if not skip_preprocessing:
        run_preprocessing(max_vocab=max_vocab)
        gc.collect()

    if not skip_classical:
        run_classical_models(quick=quick)
        gc.collect()

    if not skip_bert:
        run_bert(
            epochs     = bert_epochs,
            batch_size = bert_batch,
            quick      = quick,
        )

    log.info("\n" + "#"*68)
    log.info("  ALL PHASES COMPLETE")
    log.info(f"  Results   : {RESULTS_PATH.resolve()}")
    log.info(f"  Models    : {(BASE_WORK / 'models').resolve()}")
    log.info("#"*68)


# =============================================================================
# CLI  (use when running as !python model_a_kaggle.py)
main()

In [3]:
!zip -r working.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/train_tfidf.npz (deflated 0%)
  adding: kaggle/working/test_tfidf.npz (deflated 0%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 74%)
  adding: kaggle/working/test_ohe.npz (deflated 1%)
  adding: kaggle/working/train_ohe.npz (deflated 1%)
  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/model_a_results_20260508_153544.csv (deflated 44%)
  adding: kaggle/working/results/model_a_results_20260508_163133.csv (deflated 44%)
  adding: kaggle/working/results/model_a_supervised_results.csv (deflated 44%)
  adding: kaggle/working/results/model_a_test_result.csv (deflated 15%)
  adding: kaggle/working/results/model_a_unsupervised_results.csv (deflated 14%)
  adding: kaggle/working/preprocessing_config.json (deflated 71%)
  adding: kaggle/working/models/ (stored 0%)
  adding: kaggle/working/models/model_a/ (stored 

In [5]:
from IPython.display import FileLink
FileLink('working.zip')

/kaggle/working/working.zip